In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
import os
print(os.listdir("/kaggle/input"))

In [ ]:
import os

base = "/kaggle/input/datasets"
for name in os.listdir(base):
    print(name, "->", os.listdir(os.path.join(base, name))[:10])

In [ ]:
import os
base_fm = "/kaggle/input/datasets/organizations/zalando-research"
print(os.listdir(base_fm))

In [ ]:
fm_dir = os.path.join(base_fm, os.listdir(base_fm)[0])  # quick pick first
print("fm_dir:", fm_dir)
print(os.listdir(fm_dir)[:50])

In [ ]:
#importing all python libraries needed 
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
#gpu and hyperparameters - sets training settings - using common baseline settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

BATCH_SIZE = 64
IMG_SIZE = 224
EPOCHS = 5
LR = 1e-4
NUM_CLASSES = 10

In [ ]:
#transforms - prepossessing and augementation 
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
#fashionmnstcsv dataset - building custom pytorch dataset for my dataset
class FashionMNISTCSV(Dataset):

    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

        self.labels = self.df.iloc[:,0].to_numpy(dtype=np.int64)
        self.pixels = self.df.iloc[:,1:].to_numpy(dtype=np.uint8).reshape(-1,28,28)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        img = Image.fromarray(self.pixels[idx]).convert("RGB")
        y = int(self.labels[idx])

        if self.transform:
            img = self.transform(img)

        return img, y

In [ ]:
#load dataloader and fashionmnst, using dataloaders to handle batching shuffing and iteration 
fm_dir = "/kaggle/input/datasets/organizations/zalando-research/fashionmnist"

train_csv = f"{fm_dir}/fashion-mnist_train.csv"
test_csv  = f"{fm_dir}/fashion-mnist_test.csv"

fm_train = FashionMNISTCSV(train_csv, transform=train_tfms)
fm_test  = FashionMNISTCSV(test_csv, transform=test_tfms)

fm_train_loader = DataLoader(fm_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
fm_test_loader  = DataLoader(fm_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

fm_class_names = [
"T-shirt/top","Trouser","Pullover","Dress","Coat",
"Sandal","Shirt","Sneaker","Bag","Ankle boot"]

In [ ]:
#loading product images - using styles and id mapping for dataset strcuture 
DATA_DIR = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-small"

styles = pd.read_csv(f"{DATA_DIR}/styles.csv", on_bad_lines="skip")
images_dir = f"{DATA_DIR}/images"

styles = styles.dropna(subset=["id","articleType"])
styles["id"] = styles["id"].astype(int)

styles["img_path"] = styles["id"].astype(str).apply(
    lambda x: os.path.join(images_dir,x+".jpg")
)

styles = styles[styles["img_path"].apply(os.path.exists)]

top10 = styles["articleType"].value_counts().head(10).index #using articleType to create manageable 10 class classificiation like fashionmnst
df = styles[styles["articleType"].isin(top10)]

In [ ]:
min_count = df["articleType"].value_counts().min()

df_bal = df.groupby("articleType").sample(n=min_count, random_state=42)

label2idx = {lbl:i for i,lbl in enumerate(sorted(df_bal["articleType"].unique()))}
idx2label = {v:k for k,v in label2idx.items()}

df_bal["label"] = df_bal["articleType"].map(label2idx)

train_df,test_df = train_test_split(df_bal,test_size=0.2,
                                    stratify=df_bal["label"],
                                    random_state=42)

In [ ]:
class ProductDataset(Dataset): #productdataset

    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = Image.open(row["img_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            img = self.transform(img)

        return img,label

In [ ]:
#product dataloader 
prod_train = ProductDataset(train_df, transform=train_tfms)
prod_test  = ProductDataset(test_df, transform=test_tfms)

prod_train_loader = DataLoader(prod_train, batch_size=BATCH_SIZE, shuffle=True)
prod_test_loader  = DataLoader(prod_test, batch_size=BATCH_SIZE, shuffle=False)

prod_class_names = [idx2label[i] for i in range(10)]

In [ ]:
#resnet pretrained model
def make_resnet18(num_classes=10): #using 10 classes 

    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model.to(DEVICE)

In [ ]:
#trainiung loop 
def train_one_epoch(model,loader,optimizer,criterion):

    model.train()
    total_loss,correct,total = 0,0,0

    for x,y in loader:

        x,y = x.to(DEVICE),y.to(DEVICE)

        optimizer.zero_grad()

        out = model(x)

        loss = criterion(out,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()*x.size(0)
        correct += (out.argmax(1)==y).sum().item()
        total += x.size(0)

    return total_loss/total, correct/total

In [ ]:
#evaluation funtion
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np
import torch

@torch.no_grad()
def evaluate_all_metrics(model, loader):
    model.eval()
    ys, ps = [], []

    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        p = out.argmax(1).cpu().numpy()

        ys.append(y.numpy())
        ps.append(p)

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")

    return acc, precision, recall, f1, y_true, y_pred

In [ ]:
#fashionmnst resnet 18 training 
model_fm = make_resnet18(NUM_CLASSES)

optimizer = torch.optim.Adam(model_fm.parameters(),lr=LR)
criterion = nn.CrossEntropyLoss()

for e in range(EPOCHS):

    loss,acc = train_one_epoch(model_fm,fm_train_loader,optimizer,criterion)

    print("Epoch",e+1,"Loss",loss,"Acc",acc)

In [ ]:
#evaluation of fashionmnst
fm_acc, fm_prec, fm_rec, fm_f1, fm_y, fm_p = evaluate_all_metrics(model_fm, fm_test_loader)

print("FashionMNIST Accuracy:", fm_acc)
print("FashionMNIST Precision:", fm_prec)
print("FashionMNIST Recall:", fm_rec)
print("FashionMNIST F1:", fm_f1)

In [ ]:
#training resnet on product images dataset
model_prod = make_resnet18(NUM_CLASSES)

optimizer = torch.optim.Adam(model_prod.parameters(),lr=LR)

for e in range(EPOCHS):

    loss,acc = train_one_epoch(model_prod,prod_train_loader,optimizer,criterion)

    print("Epoch",e+1,"Loss",loss,"Acc",acc)

In [ ]:
#evaluationg product images 
prod_acc, prod_prec, prod_rec, prod_f1, prod_y, prod_p = evaluate_all_metrics(model_prod, prod_test_loader)

print("Product Images Accuracy:", prod_acc)
print("Product Images Precision:", prod_prec)
print("Product Images Recall:", prod_rec)
print("Product Images F1:", prod_f1)

In [ ]:
#results table comparing both datasets
import pandas as pd

results = pd.DataFrame([
    {
        "Dataset": "FashionMNIST",
        "Accuracy": fm_acc,
        "Precision": fm_prec,
        "Recall": fm_rec,
        "MacroF1": fm_f1
    },
    {
        "Dataset": "ProductImages",
        "Accuracy": prod_acc,
        "Precision": prod_prec,
        "Recall": prod_rec,
        "MacroF1": prod_f1
    }
])

results

FashionMNIST did better overall in comparison to ProductImages

In [ ]:
import os

shoes_root = "/kaggle/input/datasets/utkarshsaxenadn"

print("Level 1:", os.listdir(shoes_root))

for d in os.listdir(shoes_root):
    path = os.path.join(shoes_root, d)
    print("\nInside:", path)
    print(os.listdir(path)[:20])

In [ ]:
#imports
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

BATCH_SIZE = 64
IMG_SIZE = 224
EPOCHS = 5
LR = 1e-4

In [ ]:
#imagetransforms
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
#loading shoe data
shoes_base = "/kaggle/input/datasets/utkarshsaxenadn/shoes-classification-dataset-13k-images/Shoes Dataset"

train_dir = os.path.join(shoes_base, "Train")
valid_dir = os.path.join(shoes_base, "Valid")
test_dir  = os.path.join(shoes_base, "Test")

print("Train exists:", os.path.exists(train_dir))
print("Valid exists:", os.path.exists(valid_dir))
print("Test exists:", os.path.exists(test_dir))
print("Train folders:", os.listdir(train_dir)[:10])

shoes_train = ImageFolder(train_dir, transform=train_tfms)
shoes_valid = ImageFolder(valid_dir, transform=test_tfms)
shoes_test  = ImageFolder(test_dir, transform=test_tfms)

shoes_train_loader = DataLoader(shoes_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
shoes_valid_loader = DataLoader(shoes_valid, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
shoes_test_loader  = DataLoader(shoes_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

shoes_class_names = shoes_train.classes

print("Train size:", len(shoes_train))
print("Valid size:", len(shoes_valid))
print("Test size:", len(shoes_test))
print("Classes:", shoes_class_names)

In [ ]:
x, y = next(iter(shoes_train_loader))
print("x shape:", x.shape)
print("y shape:", y.shape)
print("sample labels:", y[:10])

In [ ]:
#resnet model training and evaluation
def make_resnet18(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(DEVICE)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_all_metrics(model, loader):
    model.eval()
    ys, ps = [], []

    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        p = out.argmax(1).cpu().numpy()

        ys.append(y.numpy())
        ps.append(p)

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")

    return acc, precision, recall, f1

In [ ]:
#train the model
model_shoes = make_resnet18(num_classes=len(shoes_class_names))
optimizer = torch.optim.Adam(model_shoes.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for e in range(EPOCHS):
    loss, acc = train_one_epoch(model_shoes, shoes_train_loader, optimizer, criterion)
    print(f"Epoch {e+1}/{EPOCHS} | Loss={loss:.4f} | Train Acc={acc:.4f}")

In [ ]:
#evaluation on test data
sh_acc, sh_prec, sh_rec, sh_f1 = evaluate_all_metrics(model_shoes, shoes_test_loader)

print("Shoes Accuracy:", sh_acc)
print("Shoes Precision:", sh_prec)
print("Shoes Recall:", sh_rec)
print("Shoes F1:", sh_f1)

In [ ]:
import os

jew_root = "/kaggle/input/datasets/sapnilpatel"

print("Level 1:", os.listdir(jew_root))

for d in os.listdir(jew_root):
    path = os.path.join(jew_root, d)
    print("\nInside:", path)
    print(os.listdir(path)[:20])

In [ ]:
jew_base = "/kaggle/input/datasets/sapnilpatel/tanishq-jewellery-dataset"

print("Exists:", os.path.exists(jew_base))
if os.path.exists(jew_base):
    print("Inside tanishq-jewellery-dataset:", os.listdir(jew_base)[:30])

In [ ]:
import os

jew_path = "/kaggle/input/datasets/sapnilpatel/tanishq-jewellery-dataset/Jewellery_Data"

print("Inside Jewellery_Data:", os.listdir(jew_path)[:30])

for item in os.listdir(jew_path):
    full_path = os.path.join(jew_path, item)
    print("\nItem:", item)
    print("Is dir:", os.path.isdir(full_path))
    if os.path.isdir(full_path):
        print("Contents:", os.listdir(full_path)[:20])

In [ ]:
import os
import numpy as np
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

jew_dir = "/kaggle/input/datasets/sapnilpatel/tanishq-jewellery-dataset/Jewellery_Data"

full_jew_train = ImageFolder(jew_dir, transform=train_tfms)
full_jew_test  = ImageFolder(jew_dir, transform=test_tfms)

print("Classes:", full_jew_train.classes)
print("Total images:", len(full_jew_train))

In [ ]:
#creating train test split
indices = np.arange(len(full_jew_train))
np.random.seed(42)
np.random.shuffle(indices)

train_size = int(0.8 * len(indices))
train_idx = indices[:train_size]
test_idx = indices[train_size:]

jew_train = Subset(full_jew_train, train_idx)
jew_test  = Subset(full_jew_test, test_idx)

jew_train_loader = DataLoader(jew_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
jew_test_loader  = DataLoader(jew_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

jew_class_names = full_jew_train.classes

print("Train size:", len(jew_train))
print("Test size:", len(jew_test))
print("Classes:", jew_class_names)

In [ ]:
#training resnet on jewellery dataset 
model_jew = make_resnet18(num_classes=len(jew_class_names))
optimizer = torch.optim.Adam(model_jew.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for e in range(EPOCHS):
    loss, acc = train_one_epoch(model_jew, jew_train_loader, optimizer, criterion)
    print(f"Epoch {e+1}/{EPOCHS} | Loss={loss:.4f} | Train Acc={acc:.4f}")

In [ ]:
@torch.no_grad()
def evaluate_all_metrics(model, loader):
    model.eval()

    ys, ps = [], []

    for x, y in loader:
        x = x.to(DEVICE)

        out = model(x)
        p = out.argmax(1).cpu().numpy()

        ys.append(y.numpy())
        ps.append(p)

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    return acc, precision, recall, f1, y_true, y_pred

In [ ]:
j_acc, j_prec, j_rec, j_f1, j_y, j_p = evaluate_all_metrics(model_jew, jew_test_loader)

print("Jewellery Accuracy:", j_acc)
print("Jewellery Precision:", j_prec)
print("Jewellery Recall:", j_rec)
print("Jewellery F1:", j_f1)

In [ ]:
#comparison of all datasets
import pandas as pd

results = pd.DataFrame([
    {
        "Dataset": "FashionMNIST",
        "Accuracy": fm_acc,
        "Precision": fm_prec,
        "Recall": fm_rec,
        "MacroF1": fm_f1
    },
    {
        "Dataset": "ProductImages",
        "Accuracy": prod_acc,
        "Precision": prod_prec,
        "Recall": prod_rec,
        "MacroF1": prod_f1
    },
    {
        "Dataset": "Shoes",
        "Accuracy": sh_acc,
        "Precision": sh_prec,
        "Recall": sh_rec,
        "MacroF1": sh_f1
    },
    {
        "Dataset": "Jewellery",
        "Accuracy": j_acc,
        "Precision": j_prec,
        "Recall": j_rec,
        "MacroF1": j_f1
    }
])

results

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
#converting pytorch to numpy
import numpy as np

def dataset_to_numpy(dataset):
    X, y = [], []

    for img, label in dataset:
        arr = img.numpy()                 # shape: [C, H, W]
        arr = arr.reshape(-1)             # flatten to 1D
        X.append(arr)
        y.append(label)

    X = np.array(X)
    y = np.array(y)

    return X, y

In [ ]:
from torchvision import transforms

rf_tfms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

In [ ]:
#rebuilding fashionmnst with ML
fm_train_rf = FashionMNISTCSV(train_csv, transform=rf_tfms)
fm_test_rf  = FashionMNISTCSV(test_csv, transform=rf_tfms)

X_train_fm, y_train_fm = dataset_to_numpy(fm_train_rf)
X_test_fm, y_test_fm   = dataset_to_numpy(fm_test_rf)

print("X_train shape:", X_train_fm.shape)
print("X_test shape:", X_test_fm.shape)

In [ ]:
#training random forest on dataset
rf_fm = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_fm.fit(X_train_fm, y_train_fm)

In [ ]:
#evaluation
y_pred_fm = rf_fm.predict(X_test_fm)

rf_fm_acc = accuracy_score(y_test_fm, y_pred_fm)
rf_fm_prec = precision_score(y_test_fm, y_pred_fm, average="macro")
rf_fm_rec = recall_score(y_test_fm, y_pred_fm, average="macro")
rf_fm_f1 = f1_score(y_test_fm, y_pred_fm, average="macro")

print("RF FashionMNIST Accuracy:", rf_fm_acc)
print("RF FashionMNIST Precision:", rf_fm_prec)
print("RF FashionMNIST Recall:", rf_fm_rec)
print("RF FashionMNIST F1:", rf_fm_f1)

In [ ]:
#results table
rf_results = pd.DataFrame([
    {
        "Dataset": "FashionMNIST",
        "Model": "Random Forest",
        "Accuracy": rf_fm_acc,
        "Precision": rf_fm_prec,
        "Recall": rf_fm_rec,
        "MacroF1": rf_fm_f1
    }
])

rf_results

In [ ]:
from torchvision import transforms

rf_tfms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [ ]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

In [ ]:
styles_csv = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-small/styles.csv"
img_dir = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-small/images"

df = pd.read_csv(styles_csv, on_bad_lines="skip")
df.head()

In [ ]:
df = df[["id", "masterCategory", "subCategory", "articleType"]].dropna()

valid_ids = []
for img_id in df["id"]:
    img_path = os.path.join(img_dir, f"{img_id}.jpg")
    if os.path.exists(img_path):
        valid_ids.append(img_id)

df = df[df["id"].isin(valid_ids)].copy()

top_classes = df["articleType"].value_counts().head(10).index
df = df[df["articleType"].isin(top_classes)].copy()

min_count = df["articleType"].value_counts().min()
df_bal = df.groupby("articleType").sample(n=min_count, random_state=42)

label2idx = {lbl: i for i, lbl in enumerate(sorted(df_bal["articleType"].unique()))}
idx2label = {v: k for k, v in label2idx.items()}
df_bal["label"] = df_bal["articleType"].map(label2idx)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_bal,
    test_size=0.2,
    stratify=df_bal["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Classes:", sorted(df_bal["articleType"].unique()))

In [ ]:
#recreating data classes
class ProductImagesDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(img_dir, f"{row['id']}.jpg")

        img = Image.open(img_path).convert("RGB")
        label = int(row["label"])

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
from torchvision import transforms

rf_tfms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

prod_train_rf = ProductImagesDataset(train_df, transform=rf_tfms)
prod_test_rf  = ProductImagesDataset(test_df, transform=rf_tfms)

print("Train samples:", len(prod_train_rf))
print("Test samples:", len(prod_test_rf))

In [ ]:
#convert to numpy
import numpy as np

def dataset_to_numpy(dataset):
    X, y = [], []

    for img, label in dataset:
        arr = img.numpy()
        arr = arr.reshape(-1)
        X.append(arr)
        y.append(label)

    return np.array(X), np.array(y)

X_train_prod, y_train_prod = dataset_to_numpy(prod_train_rf)
X_test_prod, y_test_prod = dataset_to_numpy(prod_test_rf)

print("X_train shape:", X_train_prod.shape)
print("X_test shape:", X_test_prod.shape)

In [ ]:
#training random forest
from sklearn.ensemble import RandomForestClassifier

rf_prod = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_prod.fit(X_train_prod, y_train_prod)

In [ ]:
#evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred_prod = rf_prod.predict(X_test_prod)

rf_prod_acc = accuracy_score(y_test_prod, y_pred_prod)
rf_prod_prec = precision_score(y_test_prod, y_pred_prod, average="macro")
rf_prod_rec = recall_score(y_test_prod, y_pred_prod, average="macro")
rf_prod_f1 = f1_score(y_test_prod, y_pred_prod, average="macro")

print("RF Product Accuracy:", rf_prod_acc)
print("RF Product Precision:", rf_prod_prec)
print("RF Product Recall:", rf_prod_rec)
print("RF Product F1:", rf_prod_f1)

In [ ]:
from torchvision import transforms

rf_tfms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [ ]:
from torchvision.datasets import ImageFolder

shoes_base = "/kaggle/input/datasets/utkarshsaxenadn/shoes-classification-dataset-13k-images/Shoes Dataset"

train_dir = f"{shoes_base}/Train"
test_dir  = f"{shoes_base}/Test"

shoes_train_rf = ImageFolder(train_dir, transform=rf_tfms)
shoes_test_rf  = ImageFolder(test_dir, transform=rf_tfms)

print("Train size:", len(shoes_train_rf))
print("Test size:", len(shoes_test_rf))
print("Classes:", shoes_train_rf.classes)

In [ ]:
import numpy as np

def dataset_to_numpy(dataset):
    X, y = [], []

    for img, label in dataset:
        arr = img.numpy()
        arr = arr.reshape(-1)
        X.append(arr)
        y.append(label)

    return np.array(X), np.array(y)

In [ ]:
#conversion
X_train_shoes, y_train_shoes = dataset_to_numpy(shoes_train_rf)
X_test_shoes, y_test_shoes   = dataset_to_numpy(shoes_test_rf)

print("X_train shape:", X_train_shoes.shape)
print("X_test shape:", X_test_shoes.shape)

In [ ]:
#training random forest
from sklearn.ensemble import RandomForestClassifier

rf_shoes = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_shoes.fit(X_train_shoes, y_train_shoes)

In [ ]:
y_pred_test_shoes = rf_shoes.predict(X_test_shoes)

rf_shoes_test_acc  = accuracy_score(y_test_shoes, y_pred_test_shoes)
rf_shoes_test_prec = precision_score(y_test_shoes, y_pred_test_shoes, average="macro")
rf_shoes_test_rec  = recall_score(y_test_shoes, y_pred_test_shoes, average="macro")
rf_shoes_test_f1   = f1_score(y_test_shoes, y_pred_test_shoes, average="macro")

print("RF Shoes Test Accuracy:", rf_shoes_test_acc)
print("RF Shoes Test Precision:", rf_shoes_test_prec)
print("RF Shoes Test Recall:", rf_shoes_test_rec)
print("RF Shoes Test F1:", rf_shoes_test_f1)

In [ ]:
#results
import pandas as pd

rf_shoes_results = pd.DataFrame([
    {
        "Dataset": "Shoes",
        "Model": "Random Forest",
        "Accuracy": rf_shoes_test_acc,
        "Precision": rf_shoes_test_prec,
        "Recall": rf_shoes_test_rec,
        "MacroF1": rf_shoes_test_f1
    }
])

rf_shoes_results

In [ ]:
from torchvision import transforms

rf_tfms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [ ]:
from torchvision.datasets import ImageFolder

jew_dir = "/kaggle/input/datasets/sapnilpatel/tanishq-jewellery-dataset/Jewellery_Data"

full_jew_rf = ImageFolder(jew_dir, transform=rf_tfms)

print("Classes:", full_jew_rf.classes)
print("Total images:", len(full_jew_rf))

In [ ]:
import numpy as np
from torch.utils.data import Subset

indices = np.arange(len(full_jew_rf))
np.random.seed(42)
np.random.shuffle(indices)

train_size = int(0.8 * len(indices))
train_idx = indices[:train_size]
test_idx = indices[train_size:]

jew_train_rf = Subset(full_jew_rf, train_idx)
jew_test_rf  = Subset(full_jew_rf, test_idx)

print("Train size:", len(jew_train_rf))
print("Test size:", len(jew_test_rf))

In [ ]:
import numpy as np

def dataset_to_numpy(dataset):
    X, y = [], []

    for img, label in dataset:
        arr = img.numpy()
        arr = arr.reshape(-1)
        X.append(arr)
        y.append(label)

    return np.array(X), np.array(y)

In [ ]:
X_train_jew, y_train_jew = dataset_to_numpy(jew_train_rf)
X_test_jew, y_test_jew   = dataset_to_numpy(jew_test_rf)

print("X_train shape:", X_train_jew.shape)
print("X_test shape:", X_test_jew.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_jew = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_jew.fit(X_train_jew, y_train_jew)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred_jew = rf_jew.predict(X_test_jew)

rf_jew_acc  = accuracy_score(y_test_jew, y_pred_jew)
rf_jew_prec = precision_score(y_test_jew, y_pred_jew, average="macro")
rf_jew_rec  = recall_score(y_test_jew, y_pred_jew, average="macro")
rf_jew_f1   = f1_score(y_test_jew, y_pred_jew, average="macro")

print("RF Jewellery Accuracy:", rf_jew_acc)
print("RF Jewellery Precision:", rf_jew_prec)
print("RF Jewellery Recall:", rf_jew_rec)
print("RF Jewellery F1:", rf_jew_f1)

In [ ]:
#comparison of random forest on all 4 datasets
import pandas as pd

rf_results_all = pd.DataFrame([
    {
        "Dataset": "FashionMNIST",
        "Model": "Random Forest",
        "Accuracy": rf_fm_acc,
        "Precision": rf_fm_prec,
        "Recall": rf_fm_rec,
        "MacroF1": rf_fm_f1
    },
    {
        "Dataset": "ProductImages",
        "Model": "Random Forest",
        "Accuracy": rf_prod_acc,
        "Precision": rf_prod_prec,
        "Recall": rf_prod_rec,
        "MacroF1": rf_prod_f1
    },
    {
        "Dataset": "Shoes",
        "Model": "Random Forest",
        "Accuracy": rf_shoes_test_acc,
        "Precision": rf_shoes_test_prec,
        "Recall": rf_shoes_test_rec,
        "MacroF1": rf_shoes_test_f1
    },
    {
        "Dataset": "Jewellery",
        "Model": "Random Forest",
        "Accuracy": rf_jew_acc,
        "Precision": rf_jew_prec,
        "Recall": rf_jew_rec,
        "MacroF1": rf_jew_f1
    }
])

rf_results_all

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import mode

In [ ]:
from torchvision import transforms

km_tfms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [ ]:
fm_train_km = FashionMNISTCSV(train_csv, transform=km_tfms)
fm_test_km  = FashionMNISTCSV(test_csv, transform=km_tfms)

print("Train size:", len(fm_train_km))
print("Test size:", len(fm_test_km))

In [ ]:
#numpy conversion
def dataset_to_numpy(dataset):
    X, y = [], []

    for img, label in dataset:
        arr = img.numpy()
        arr = arr.reshape(-1)
        X.append(arr)
        y.append(label)

    return np.array(X), np.array(y)

In [ ]:
X_train_fm_km, y_train_fm_km = dataset_to_numpy(fm_train_km)
X_test_fm_km, y_test_fm_km   = dataset_to_numpy(fm_test_km)

print("X_train shape:", X_train_fm_km.shape)
print("X_test shape:", X_test_fm_km.shape)

In [ ]:
#10 classes to 10 clusters
kmeans_fm = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=10
)

kmeans_fm.fit(X_train_fm_km)

In [ ]:
#cluster to label mapping
train_clusters = kmeans_fm.predict(X_train_fm_km)

cluster_to_label = {}

for i in range(10):
    labels_in_cluster = y_train_fm_km[train_clusters == i]
    if len(labels_in_cluster) == 0:
        cluster_to_label[i] = -1
    else:
        cluster_to_label[i] = mode(labels_in_cluster, keepdims=True).mode[0]

print(cluster_to_label)

In [ ]:
test_clusters = kmeans_fm.predict(X_test_fm_km)

y_pred_fm_km = np.array([cluster_to_label[c] for c in test_clusters])

In [ ]:
#evalution for fashionmnst
km_fm_acc  = accuracy_score(y_test_fm_km, y_pred_fm_km)
km_fm_prec = precision_score(y_test_fm_km, y_pred_fm_km, average="macro", zero_division=0)
km_fm_rec  = recall_score(y_test_fm_km, y_pred_fm_km, average="macro", zero_division=0)
km_fm_f1   = f1_score(y_test_fm_km, y_pred_fm_km, average="macro", zero_division=0)

print("KMeans FashionMNIST Accuracy:", km_fm_acc)
print("KMeans FashionMNIST Precision:", km_fm_prec)
print("KMeans FashionMNIST Recall:", km_fm_rec)
print("KMeans FashionMNIST F1:", km_fm_f1)

In [ ]:
from torchvision import transforms

km_tfms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [ ]:
prod_train_km = ProductImagesDataset(train_df, transform=km_tfms)
prod_test_km  = ProductImagesDataset(test_df, transform=km_tfms)

print("Train samples:", len(prod_train_km))
print("Test samples:", len(prod_test_km))

In [ ]:
import numpy as np

def dataset_to_numpy(dataset):
    X, y = [], []

    for img, label in dataset:
        arr = img.numpy()
        arr = arr.reshape(-1)
        X.append(arr)
        y.append(label)

    return np.array(X), np.array(y)

In [ ]:
X_train_prod_km, y_train_prod_km = dataset_to_numpy(prod_train_km)
X_test_prod_km, y_test_prod_km   = dataset_to_numpy(prod_test_km)

print("X_train shape:", X_train_prod_km.shape)
print("X_test shape:", X_test_prod_km.shape)

In [ ]:
from sklearn.cluster import KMeans

kmeans_prod = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=10
)

kmeans_prod.fit(X_train_prod_km)

In [ ]:
#cluster to label mapping
from scipy.stats import mode

train_clusters_prod = kmeans_prod.predict(X_train_prod_km)

cluster_to_label_prod = {}

for i in range(10):
    labels_in_cluster = y_train_prod_km[train_clusters_prod == i]
    if len(labels_in_cluster) == 0:
        cluster_to_label_prod[i] = -1
    else:
        cluster_to_label_prod[i] = mode(labels_in_cluster, keepdims=True).mode[0]

print(cluster_to_label_prod)

In [ ]:
test_clusters_prod = kmeans_prod.predict(X_test_prod_km)
y_pred_prod_km = np.array([cluster_to_label_prod[c] for c in test_clusters_prod])

In [ ]:
#evaluation for product images
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

km_prod_acc  = accuracy_score(y_test_prod_km, y_pred_prod_km)
km_prod_prec = precision_score(y_test_prod_km, y_pred_prod_km, average="macro", zero_division=0)
km_prod_rec  = recall_score(y_test_prod_km, y_pred_prod_km, average="macro", zero_division=0)
km_prod_f1   = f1_score(y_test_prod_km, y_pred_prod_km, average="macro", zero_division=0)

print("KMeans Product Accuracy:", km_prod_acc)
print("KMeans Product Precision:", km_prod_prec)
print("KMeans Product Recall:", km_prod_rec)
print("KMeans Product F1:", km_prod_f1)

In [ ]:
from torchvision import transforms

km_tfms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [ ]:
from torchvision.datasets import ImageFolder

shoes_base = "/kaggle/input/datasets/utkarshsaxenadn/shoes-classification-dataset-13k-images/Shoes Dataset"

train_dir = f"{shoes_base}/Train"
test_dir  = f"{shoes_base}/Test"

shoes_train_km = ImageFolder(train_dir, transform=km_tfms)
shoes_test_km  = ImageFolder(test_dir, transform=km_tfms)

print("Train size:", len(shoes_train_km))
print("Test size:", len(shoes_test_km))
print("Classes:", shoes_train_km.classes)

In [ ]:
import numpy as np

def dataset_to_numpy(dataset):
    X, y = [], []

    for img, label in dataset:
        arr = img.numpy()
        arr = arr.reshape(-1)
        X.append(arr)
        y.append(label)

    return np.array(X), np.array(y)

In [ ]:
X_train_shoes_km, y_train_shoes_km = dataset_to_numpy(shoes_train_km)
X_test_shoes_km, y_test_shoes_km   = dataset_to_numpy(shoes_test_km)

print("X_train shape:", X_train_shoes_km.shape)
print("X_test shape:", X_test_shoes_km.shape)

In [ ]:
from sklearn.cluster import KMeans

kmeans_shoes = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

kmeans_shoes.fit(X_train_shoes_km)

In [ ]:
from scipy.stats import mode

train_clusters_shoes = kmeans_shoes.predict(X_train_shoes_km)

cluster_to_label_shoes = {}

for i in range(5):
    labels_in_cluster = y_train_shoes_km[train_clusters_shoes == i]
    if len(labels_in_cluster) == 0:
        cluster_to_label_shoes[i] = -1
    else:
        cluster_to_label_shoes[i] = mode(labels_in_cluster, keepdims=True).mode[0]

print(cluster_to_label_shoes)

In [ ]:
test_clusters_shoes = kmeans_shoes.predict(X_test_shoes_km)
y_pred_test_shoes_km = np.array([cluster_to_label_shoes[c] for c in test_clusters_shoes])

In [ ]:
#evaluation of shoes dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

km_shoes_acc  = accuracy_score(y_test_shoes_km, y_pred_test_shoes_km)
km_shoes_prec = precision_score(y_test_shoes_km, y_pred_test_shoes_km, average="macro", zero_division=0)
km_shoes_rec  = recall_score(y_test_shoes_km, y_pred_test_shoes_km, average="macro", zero_division=0)
km_shoes_f1   = f1_score(y_test_shoes_km, y_pred_test_shoes_km, average="macro", zero_division=0)

print("KMeans Shoes Accuracy:", km_shoes_acc)
print("KMeans Shoes Precision:", km_shoes_prec)
print("KMeans Shoes Recall:", km_shoes_rec)
print("KMeans Shoes F1:", km_shoes_f1)

In [ ]:
from torchvision import transforms

km_tfms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [ ]:
from torchvision.datasets import ImageFolder

jew_dir = "/kaggle/input/datasets/sapnilpatel/tanishq-jewellery-dataset/Jewellery_Data"

full_jew_km = ImageFolder(jew_dir, transform=km_tfms)

print("Classes:", full_jew_km.classes)
print("Total images:", len(full_jew_km))

In [ ]:
import numpy as np
from torch.utils.data import Subset

indices = np.arange(len(full_jew_km))
np.random.seed(42)
np.random.shuffle(indices)

train_size = int(0.8 * len(indices))
train_idx = indices[:train_size]
test_idx = indices[train_size:]

jew_train_km = Subset(full_jew_km, train_idx)
jew_test_km  = Subset(full_jew_km, test_idx)

print("Train size:", len(jew_train_km))
print("Test size:", len(jew_test_km))

In [ ]:
def dataset_to_numpy(dataset):
    X, y = [], []

    for img, label in dataset:
        arr = img.numpy()
        arr = arr.reshape(-1)
        X.append(arr)
        y.append(label)

    return np.array(X), np.array(y)

In [ ]:
X_train_jew_km, y_train_jew_km = dataset_to_numpy(jew_train_km)
X_test_jew_km, y_test_jew_km   = dataset_to_numpy(jew_test_km)

print("X_train shape:", X_train_jew_km.shape)
print("X_test shape:", X_test_jew_km.shape)

In [ ]:
from sklearn.cluster import KMeans

kmeans_jew = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

kmeans_jew.fit(X_train_jew_km)

In [ ]:
from scipy.stats import mode

train_clusters_jew = kmeans_jew.predict(X_train_jew_km)

cluster_to_label_jew = {}

for i in range(2):
    labels_in_cluster = y_train_jew_km[train_clusters_jew == i]
    if len(labels_in_cluster) == 0:
        cluster_to_label_jew[i] = -1
    else:
        cluster_to_label_jew[i] = mode(labels_in_cluster, keepdims=True).mode[0]

print(cluster_to_label_jew)

In [ ]:
test_clusters_jew = kmeans_jew.predict(X_test_jew_km)
y_pred_test_jew_km = np.array([cluster_to_label_jew[c] for c in test_clusters_jew])

In [ ]:
#evaluation of jewellery dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

km_jew_acc  = accuracy_score(y_test_jew_km, y_pred_test_jew_km)
km_jew_prec = precision_score(y_test_jew_km, y_pred_test_jew_km, average="macro", zero_division=0)
km_jew_rec  = recall_score(y_test_jew_km, y_pred_test_jew_km, average="macro", zero_division=0)
km_jew_f1   = f1_score(y_test_jew_km, y_pred_test_jew_km, average="macro", zero_division=0)

print("KMeans Jewellery Accuracy:", km_jew_acc)
print("KMeans Jewellery Precision:", km_jew_prec)
print("KMeans Jewellery Recall:", km_jew_rec)
print("KMeans Jewellery F1:", km_jew_f1)

In [ ]:
#comparison table for all 4 datasets (k-means)
import pandas as pd

km_results_all = pd.DataFrame([
    {
        "Dataset": "FashionMNIST",
        "Model": "KMeans",
        "Accuracy": km_fm_acc,
        "Precision": km_fm_prec,
        "Recall": km_fm_rec,
        "MacroF1": km_fm_f1
    },
    {
        "Dataset": "ProductImages",
        "Model": "KMeans",
        "Accuracy": km_prod_acc,
        "Precision": km_prod_prec,
        "Recall": km_prod_rec,
        "MacroF1": km_prod_f1
    },
    {
        "Dataset": "Shoes",
        "Model": "KMeans",
        "Accuracy": km_shoes_acc,
        "Precision": km_shoes_prec,
        "Recall": km_shoes_rec,
        "MacroF1": km_shoes_f1
    },
    {
        "Dataset": "Jewellery",
        "Model": "KMeans",
        "Accuracy": km_jew_acc,
        "Precision": km_jew_prec,
        "Recall": km_jew_rec,
        "MacroF1": km_jew_f1
    }
])

km_results_all.round(4)

In [ ]:
#results table for each dataset (cnn vs random forest vs k means)
import pandas as pd

all_results = pd.DataFrame([
    # FashionMNIST dataset
    {
        "Dataset": "FashionMNIST",
        "Model": "ResNet18",
        "Accuracy": fm_acc,
        "Precision": fm_prec,
        "Recall": fm_rec,
        "MacroF1": fm_f1
    },
    {
        "Dataset": "FashionMNIST",
        "Model": "Random Forest",
        "Accuracy": rf_fm_acc,
        "Precision": rf_fm_prec,
        "Recall": rf_fm_rec,
        "MacroF1": rf_fm_f1
    },
    {
        "Dataset": "FashionMNIST",
        "Model": "KMeans",
        "Accuracy": km_fm_acc,
        "Precision": km_fm_prec,
        "Recall": km_fm_rec,
        "MacroF1": km_fm_f1
    },

    # Product Images dataset
    {
        "Dataset": "ProductImages",
        "Model": "ResNet18",
        "Accuracy": prod_acc,
        "Precision": prod_prec,
        "Recall": prod_rec,
        "MacroF1": prod_f1
    },
    {
        "Dataset": "ProductImages",
        "Model": "Random Forest",
        "Accuracy": rf_prod_acc,
        "Precision": rf_prod_prec,
        "Recall": rf_prod_rec,
        "MacroF1": rf_prod_f1
    },
    {
        "Dataset": "ProductImages",
        "Model": "KMeans",
        "Accuracy": km_prod_acc,
        "Precision": km_prod_prec,
        "Recall": km_prod_rec,
        "MacroF1": km_prod_f1
    },

    # Shoes dataset
    {
        "Dataset": "Shoes",
        "Model": "ResNet18",
        "Accuracy": sh_acc,
        "Precision": sh_prec,
        "Recall": sh_rec,
        "MacroF1": sh_f1
    },
    {
        "Dataset": "Shoes",
        "Model": "Random Forest",
        "Accuracy": rf_shoes_test_acc,
        "Precision": rf_shoes_test_prec,
        "Recall": rf_shoes_test_rec,
        "MacroF1": rf_shoes_test_f1
    },
    {
        "Dataset": "Shoes",
        "Model": "KMeans",
        "Accuracy": km_shoes_acc,
        "Precision": km_shoes_prec,
        "Recall": km_shoes_rec,
        "MacroF1": km_shoes_f1
    },

    # Jewellery dataset
    {
        "Dataset": "Jewellery",
        "Model": "ResNet18",
        "Accuracy": j_acc,
        "Precision": j_prec,
        "Recall": j_rec,
        "MacroF1": j_f1
    },
    {
        "Dataset": "Jewellery",
        "Model": "Random Forest",
        "Accuracy": rf_jew_acc,
        "Precision": rf_jew_prec,
        "Recall": rf_jew_rec,
        "MacroF1": rf_jew_f1
    },
    {
        "Dataset": "Jewellery",
        "Model": "KMeans",
        "Accuracy": km_jew_acc,
        "Precision": km_jew_prec,
        "Recall": km_jew_rec,
        "MacroF1": km_jew_f1
    }
])

all_results.round(4)

In [ ]:
# Performance Comparison Graphs


import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Use your existing final results table
results_plot = all_results.copy()

metrics = ["Accuracy", "Precision", "Recall", "MacroF1"]

for metric in metrics:
    pivot = results_plot.pivot(index="Dataset", columns="Model", values=metric)

    ax = pivot.plot(kind="bar", figsize=(10, 5))
    plt.title(f"{metric} Comparison Across Models and Datasets")
    plt.xlabel("Dataset")
    plt.ylabel(metric)
    plt.ylim(0, 1.05)
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Model")
    plt.tight_layout()
    plt.show()

In [ ]:
# Confusion Matrix Function
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def plot_confusion(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    )

    disp.plot(xticks_rotation=45)
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
#Confusion matrix for each dataset
# FashionMNIST
plot_confusion(
    fm_y,
    fm_p,
    range(10),
    "Confusion Matrix - FashionMNIST"
)

# Product Images
plot_confusion(
    prod_y,
    prod_p,
    range(len(prod_class_names)),
    "Confusion Matrix - Product Images"
)

# Shoes
plot_confusion(
    y_test_shoes_km,
    y_pred_test_shoes_km,
    range(len(shoes_class_names)),
    "Confusion Matrix - Shoes"
)

# Jewellery
plot_confusion(
    y_test_jew_km,
    y_pred_test_jew_km,
    range(len(jew_class_names)),
    "Confusion Matrix - Jewellery"
)